# Análisis Exploratorio de Datos (EDA) - Predicción de Shock Hemorrágico

**Proyecto de Grado - Fase 1**

## Contenido

1. Configuración Inicial
2. Carga y Descripción del Dataset
3. Análisis General del Dataset
4. Tipificación de Variables
5. Análisis Univariado
   - Variables Categóricas/Binarias
   - Variables Numéricas
6. Análisis Bivariado
   - Categóricas vs Target
   - Numéricas vs Target
7. Análisis Multivariado
   - Correlaciones
   - PCA (Reducción de Dimensionalidad)
8. Feature Engineering Exploratorio
9. Conclusiones y Recomendaciones

---

**Objetivo**: Realizar un análisis exploratorio completo del dataset clínico para identificar patrones, relaciones y variables predictoras del shock hemorrágico.

**Dataset**: ~1300 registros, 22 variables (mayormente categóricas), objetivo binario (shock sí/no)

## 1. Configuración Inicial

In [ ]:
# Importar librerías
import sys
import os
import warnings
warnings.filterwarnings('ignore')

# Agregar directorio raíz al path
sys.path.append('..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from scipy.stats import chi2_contingency, mannwhitneyu, ttest_ind

# Módulos del proyecto
from src.analysis.eda import ShockEDAAnalyzer
from src.analysis.eda_visualizations import EDAVisualizer
from src.pipelines.eda_pipeline import EDA_Pipeline

# Configuración de visualización
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette("husl")
%matplotlib inline

# Configuración de pandas
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
pd.set_option('display.float_format', '{:.4f}'.format)

print("✓ Configuración completada")
print(f"✓ Python {sys.version.split()[0]}")
print(f"✓ Pandas {pd.__version__}")
print(f"✓ NumPy {np.__version__}")

## 2. Carga y Descripción del Dataset

In [ ]:
# Configuración de rutas
DATA_PATH = "../data/processed/shock.csv"
OUTPUT_DIR = "../reports/eda"

# Crear directorio de salida
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f"📁 Ruta del dataset: {DATA_PATH}")
print(f"📁 Directorio de salida: {OUTPUT_DIR}")

In [ ]:
# Cargar datos
df = pd.read_csv(DATA_PATH)

print(f"✓ Datos cargados exitosamente")
print(f"\nDimensiones: {df.shape[0]} filas x {df.shape[1]} columnas")

In [ ]:
# Primeras filas
print("Primeras 5 filas del dataset:\n")
df.head()

In [ ]:
# Información general
print("Información del dataset:\n")
df.info()

In [ ]:
# Tipos de datos
print("Distribución de tipos de datos:\n")
print(df.dtypes.value_counts())

## 3. Análisis General del Dataset

### A. Dimensiones y Estructura

In [ ]:
# Estadísticas básicas
print("="*70)
print("ESTADÍSTICAS BÁSICAS DEL DATASET")
print("="*70)

print(f"\n📊 Dimensiones:")
print(f"  • Número de registros (filas): {df.shape[0]:,}")
print(f"  • Número de variables (columnas): {df.shape[1]}")

print(f"\n🔢 Tipos de datos:")
for dtype, count in df.dtypes.value_counts().items():
    print(f"  • {dtype}: {count} columnas")

print(f"\n📝 Valores únicos por variable:")
nunique = df.nunique().sort_values(ascending=False)
print(f"  • Variable con más valores únicos: {nunique.index[0]} ({nunique.iloc[0]} valores)")
print(f"  • Variable con menos valores únicos: {nunique.index[-1]} ({nunique.iloc[-1]} valores)")

In [ ]:
# Duplicados
n_duplicates = df.duplicated().sum()
print(f"\n🔍 Duplicados: {n_duplicates}")

if n_duplicates > 0:
    print(f"  ⚠️ Se encontraron {n_duplicates} filas duplicadas")
    print(f"  Duplicados representan: {n_duplicates/len(df)*100:.2f}% del dataset")
else:
    print("  ✓ No se encontraron filas duplicadas")

### B. Valores Faltantes

In [ ]:
# Análisis de valores nulos
missing = df.isnull().sum()
missing_pct = (missing / len(df)) * 100
missing_df = pd.DataFrame({
    'Columna': missing.index,
    'Valores_Faltantes': missing.values,
    'Porcentaje': missing_pct.values
}).sort_values('Valores_Faltantes', ascending=False)

# Filtrar solo columnas con valores faltantes
missing_df = missing_df[missing_df['Valores_Faltantes'] > 0]

print("="*70)
print("ANÁLISIS DE VALORES FALTANTES")
print("="*70)

total_nulls = missing.sum()
print(f"\nTotal de valores faltantes: {total_nulls:,}")
print(f"Porcentaje total: {(total_nulls / (df.shape[0] * df.shape[1])) * 100:.2f}%")

if len(missing_df) > 0:
    print(f"\nColumnas con valores faltantes: {len(missing_df)}/{df.shape[1]}\n")
    print(missing_df.to_string(index=False))
else:
    print("\n✓ No hay valores faltantes en el dataset")

In [ ]:
# Visualizar valores faltantes (si existen)
if len(missing_df) > 0 and len(missing_df) <= 20:
    fig, ax = plt.subplots(figsize=(10, max(6, len(missing_df) * 0.4)))
    
    ax.barh(missing_df['Columna'], missing_df['Valores_Faltantes'], color='coral')
    ax.set_xlabel('Número de Valores Faltantes', fontsize=12)
    ax.set_ylabel('Variable', fontsize=12)
    ax.set_title('Variables con Valores Faltantes', fontsize=14, fontweight='bold')
    ax.grid(True, alpha=0.3, axis='x')
    
    plt.tight_layout()
    plt.show()
elif len(missing_df) == 0:
    print("No hay valores faltantes para visualizar")

### C. Distribución de la Variable Objetivo

In [ ]:
# Detectar columna objetivo
target_col = next((c for c in df.columns if 'shock' in c.lower()), None)

if target_col:
    print(f"✓ Columna objetivo detectada: '{target_col}'")
else:
    print("⚠️ No se detectó columna objetivo")
    target_col = input("Ingrese el nombre de la columna objetivo: ")

In [ ]:
# Análisis de distribución del target
if target_col and target_col in df.columns:
    print("="*70)
    print(f"DISTRIBUCIÓN DE LA VARIABLE OBJETIVO: {target_col}")
    print("="*70)
    
    target_counts = df[target_col].value_counts().sort_index()
    target_pct = df[target_col].value_counts(normalize=True).sort_index() * 100
    
    print("\nConteo por clase:")
    for clase, count in target_counts.items():
        pct = target_pct[clase]
        print(f"  • Clase {clase}: {count:,} registros ({pct:.2f}%)")
    
    # Calcular ratio de desbalance
    if len(target_counts) == 2:
        minority_class = target_counts.min()
        majority_class = target_counts.max()
        imbalance_ratio = minority_class / majority_class
        
        print(f"\n📊 Análisis de Balance:")
        print(f"  • Clase minoritaria: {minority_class:,} registros")
        print(f"  • Clase mayoritaria: {majority_class:,} registros")
        print(f"  • Ratio de desbalance: {imbalance_ratio:.4f}")
        
        if imbalance_ratio < 0.5:
            print(f"  ⚠️ Dataset DESBALANCEADO (requiere técnicas como SMOTE)")
        elif imbalance_ratio < 0.8:
            print(f"  ⚠️ Dataset MODERADAMENTE desbalanceado")
        else:
            print(f"  ✓ Dataset relativamente balanceado")

In [ ]:
# Visualización de la distribución del target
if target_col and target_col in df.columns:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Gráfico de barras
    target_counts = df[target_col].value_counts().sort_index()
    colors = ['#3498db', '#e74c3c']
    
    axes[0].bar(target_counts.index.astype(str), target_counts.values, color=colors)
    axes[0].set_xlabel('Clase', fontsize=12)
    axes[0].set_ylabel('Frecuencia', fontsize=12)
    axes[0].set_title(f'Distribución de {target_col}', fontsize=14, fontweight='bold')
    axes[0].grid(True, alpha=0.3, axis='y')
    
    # Añadir etiquetas
    for i, (idx, val) in enumerate(target_counts.items()):
        pct = (val / len(df)) * 100
        axes[0].text(i, val, f'{val}\n({pct:.1f}%)', 
                    ha='center', va='bottom', fontweight='bold', fontsize=11)
    
    # Gráfico de pastel
    axes[1].pie(target_counts.values, 
               labels=[f'Clase {i}' for i in target_counts.index],
               autopct='%1.1f%%', startangle=90, colors=colors,
               textprops={'fontsize': 11, 'fontweight': 'bold'})
    axes[1].set_title(f'Proporción de {target_col}', fontsize=14, fontweight='bold')
    
    plt.tight_layout()
    plt.show()

## 4. Tipificación de Variables

Clasificamos las variables en:
- **Numéricas (continuas)**: Variables con muchos valores únicos (>10)
- **Binarias**: Variables con exactamente 2 valores únicos
- **Categóricas**: Variables con pocos valores únicos (≤10)

In [ ]:
# Excluir columnas no predictoras
exclude_cols = ['CODIGO', 'codigo', target_col]

# Variables numéricas
numerical_cols = [
    col for col in df.select_dtypes(include=[np.number]).columns
    if col not in exclude_cols and df[col].nunique() > 10
]

# Variables binarias
binary_cols = [
    col for col in df.columns
    if col not in exclude_cols and df[col].nunique() == 2
]

# Variables categóricas
categorical_cols = [
    col for col in df.columns
    if col not in exclude_cols + numerical_cols + binary_cols
]

print("="*70)
print("TIPIFICACIÓN DE VARIABLES")
print("="*70)

print(f"\n📊 Variables numéricas (continuas): {len(numerical_cols)}")
if len(numerical_cols) > 0:
    print(f"  Variables: {numerical_cols}")

print(f"\n🔢 Variables binarias: {len(binary_cols)}")
if len(binary_cols) > 0:
    if len(binary_cols) <= 15:
        print(f"  Variables: {binary_cols}")
    else:
        print(f"  Primeras 15: {binary_cols[:15]}")
        print(f"  ... y {len(binary_cols) - 15} más")

print(f"\n📁 Variables categóricas: {len(categorical_cols)}")
if len(categorical_cols) > 0:
    print(f"  Variables: {categorical_cols}")

In [ ]:
# Crear tabla de tipificación
type_summary = []

for col in numerical_cols:
    type_summary.append({
        'Variable': col,
        'Tipo': 'Numérica',
        'N_Valores_Únicos': df[col].nunique(),
        'Valores_Nulos': df[col].isnull().sum()
    })

for col in binary_cols:
    type_summary.append({
        'Variable': col,
        'Tipo': 'Binaria',
        'N_Valores_Únicos': df[col].nunique(),
        'Valores_Nulos': df[col].isnull().sum()
    })

for col in categorical_cols:
    type_summary.append({
        'Variable': col,
        'Tipo': 'Categórica',
        'N_Valores_Únicos': df[col].nunique(),
        'Valores_Nulos': df[col].isnull().sum()
    })

type_df = pd.DataFrame(type_summary)

print("\nResumen de Tipificación:\n")
type_df

## 5. Análisis Univariado

### A. Variables Categóricas y Binarias

In [ ]:
# Análisis de frecuencias para categóricas/binarias
all_categorical = categorical_cols + binary_cols

print("="*70)
print("ANÁLISIS UNIVARIADO - VARIABLES CATEGÓRICAS/BINARIAS")
print("="*70)

freq_data = []

for col in all_categorical[:10]:  # Primeras 10 para no saturar
    print(f"\n{col}:")
    freq = df[col].value_counts()
    freq_pct = df[col].value_counts(normalize=True) * 100
    
    for val in freq.index:
        print(f"  • {val}: {freq[val]:,} ({freq_pct[val]:.2f}%)")
        freq_data.append({
            'Variable': col,
            'Valor': val,
            'Frecuencia': freq[val],
            'Porcentaje': freq_pct[val]
        })

if len(all_categorical) > 10:
    print(f"\n... y {len(all_categorical) - 10} variables más")

### B. Variables Numéricas

In [ ]:
# Estadísticas descriptivas de variables numéricas
if len(numerical_cols) > 0:
    print("="*70)
    print("ANÁLISIS UNIVARIADO - VARIABLES NUMÉRICAS")
    print("="*70)
    
    desc_stats = df[numerical_cols].describe().T
    
    # Agregar métricas adicionales
    desc_stats['skewness'] = df[numerical_cols].skew()
    desc_stats['kurtosis'] = df[numerical_cols].kurtosis()
    desc_stats['nulls'] = df[numerical_cols].isnull().sum()
    
    print("\nEstadísticas Descriptivas:\n")
    desc_stats
else:
    print("No hay variables numéricas para analizar")

In [ ]:
# Visualizar distribuciones de variables numéricas
if len(numerical_cols) > 0:
    n_num = len(numerical_cols)
    n_cols = min(n_num, 3)
    n_rows = (n_num + n_cols - 1) // n_cols
    
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(6*n_cols, 4*n_rows))
    axes = axes.flatten() if n_rows * n_cols > 1 else [axes]
    
    for idx, col in enumerate(numerical_cols):
        ax = axes[idx]
        
        # Histograma
        ax.hist(df[col].dropna(), bins=30, color='steelblue', edgecolor='black', alpha=0.7)
        ax.set_xlabel(col, fontsize=11)
        ax.set_ylabel('Frecuencia', fontsize=11)
        ax.set_title(f'Distribución de {col}', fontsize=12, fontweight='bold')
        ax.grid(True, alpha=0.3)
        
        # Añadir estadísticas
        mean_val = df[col].mean()
        median_val = df[col].median()
        ax.axvline(mean_val, color='red', linestyle='--', linewidth=2, label=f'Media: {mean_val:.2f}')
        ax.axvline(median_val, color='green', linestyle='--', linewidth=2, label=f'Mediana: {median_val:.2f}')
        ax.legend(fontsize=9)
    
    # Ocultar ejes sobrantes
    for idx in range(len(numerical_cols), len(axes)):
        axes[idx].axis('off')
    
    plt.tight_layout()
    plt.show()

In [ ]:
# Boxplots de variables numéricas
if len(numerical_cols) > 0:
    fig, ax = plt.subplots(figsize=(12, 6))
    
    df[numerical_cols].boxplot(ax=ax, vert=False, patch_artist=True,
                                boxprops=dict(facecolor='lightblue', color='blue'),
                                medianprops=dict(color='red', linewidth=2),
                                whiskerprops=dict(color='blue'),
                                capprops=dict(color='blue'))
    
    ax.set_xlabel('Valor', fontsize=12)
    ax.set_ylabel('Variable', fontsize=12)
    ax.set_title('Boxplots de Variables Numéricas (Detección de Outliers)', 
                fontsize=14, fontweight='bold')
    ax.grid(True, alpha=0.3, axis='x')
    
    plt.tight_layout()
    plt.show()

## 6. Análisis Bivariado

### A. Variables Categóricas vs Target

Utilizamos:
- **Test Chi-cuadrado**: Para probar independencia
- **Cramer's V**: Para medir la fuerza de la asociación (0 = sin asociación, 1 = asociación perfecta)

In [ ]:
# Función para calcular Cramer's V
def cramers_v(x, y):
    """Calcula Cramer's V para asociación categórica."""
    confusion_matrix = pd.crosstab(x, y)
    
    if confusion_matrix.size == 0 or confusion_matrix.shape[0] == 1 or confusion_matrix.shape[1] == 1:
        return 0.0
    
    chi2 = chi2_contingency(confusion_matrix)[0]
    n = confusion_matrix.sum().sum()
    phi2 = chi2 / n
    r, k = confusion_matrix.shape
    
    phi2corr = max(0, phi2 - ((k-1)*(r-1))/(n-1))
    rcorr = r - ((r-1)**2)/(n-1)
    kcorr = k - ((k-1)**2)/(n-1)
    denom = min(kcorr-1, rcorr-1)
    
    if denom <= 0:
        return 0.0
    
    return np.sqrt(phi2corr / denom)

print("✓ Función cramers_v definida")

In [ ]:
# Análisis categóricas vs target
if target_col and len(all_categorical) > 0:
    print("="*70)
    print("ANÁLISIS BIVARIADO - CATEGÓRICAS vs TARGET")
    print("="*70)
    
    cat_results = []
    
    for col in all_categorical:
        try:
            # Chi-cuadrado
            contingency = pd.crosstab(df[col], df[target_col])
            chi2, p_value, dof, expected = chi2_contingency(contingency)
            
            # Cramer's V
            cramers = cramers_v(df[col], df[target_col])
            
            cat_results.append({
                'Variable': col,
                'Chi2': chi2,
                'p_value': p_value,
                'Cramers_V': cramers,
                'Significativa': 'Sí' if p_value < 0.05 else 'No'
            })
        except Exception as e:
            print(f"Error procesando {col}: {e}")
    
    cat_results_df = pd.DataFrame(cat_results).sort_values('Cramers_V', ascending=False)
    
    print(f"\nTotal de variables analizadas: {len(cat_results_df)}")
    sig_count = (cat_results_df['p_value'] < 0.05).sum()
    print(f"Variables significativas (p < 0.05): {sig_count}")
    
    print(f"\n✓ Top 10 asociaciones más fuertes (Cramer's V):\n")
    cat_results_df.head(10)
else:
    print("No hay variables categóricas o target no definido")

In [ ]:
# Visualizar top asociaciones categóricas
if 'cat_results_df' in locals() and not cat_results_df.empty:
    top_cat = cat_results_df.head(15)
    
    fig, ax = plt.subplots(figsize=(10, max(6, len(top_cat) * 0.4)))
    
    colors = ['#2ecc71' if p < 0.05 else '#95a5a6' for p in top_cat['p_value']]
    
    ax.barh(top_cat['Variable'], top_cat['Cramers_V'], color=colors)
    ax.set_xlabel("Cramer's V", fontsize=12)
    ax.set_ylabel('Variable', fontsize=12)
    ax.set_title("Top 15 Asociaciones Categóricas con Target", fontsize=14, fontweight='bold')
    ax.grid(True, alpha=0.3, axis='x')
    ax.set_xlim(0, 1)
    
    # Leyenda
    from matplotlib.patches import Patch
    legend_elements = [
        Patch(facecolor='#2ecc71', label='p < 0.05 (Significativa)'),
        Patch(facecolor='#95a5a6', label='p ≥ 0.05 (No significativa)')
    ]
    ax.legend(handles=legend_elements, loc='best')
    
    plt.tight_layout()
    plt.show()

### B. Variables Numéricas vs Target

Utilizamos:
- **Test Mann-Whitney U**: Test no paramétrico para comparar dos grupos
- **Test t de Student**: Test paramétrico (asume normalidad)
- **Cohen's d**: Medida del tamaño del efecto

In [ ]:
# Análisis numéricas vs target
if target_col and len(numerical_cols) > 0:
    print("="*70)
    print("ANÁLISIS BIVARIADO - NUMÉRICAS vs TARGET")
    print("="*70)
    
    num_results = []
    
    for col in numerical_cols:
        try:
            # Separar por grupos
            group_0 = df[df[target_col] == 0][col].dropna()
            group_1 = df[df[target_col] == 1][col].dropna()
            
            if len(group_0) < 2 or len(group_1) < 2:
                continue
            
            # Mann-Whitney U test
            u_stat, p_value_mw = mannwhitneyu(group_0, group_1, alternative='two-sided')
            
            # T-test
            t_stat, p_value_t = ttest_ind(group_0, group_1, equal_var=False)
            
            # Cohen's d
            mean_0, mean_1 = group_0.mean(), group_1.mean()
            std_0, std_1 = group_0.std(), group_1.std()
            pooled_std = np.sqrt(((len(group_0)-1)*std_0**2 + (len(group_1)-1)*std_1**2) / 
                                (len(group_0)+len(group_1)-2))
            cohens_d = (mean_1 - mean_0) / pooled_std if pooled_std > 0 else 0
            
            num_results.append({
                'Variable': col,
                'Media_NoShock': mean_0,
                'Media_Shock': mean_1,
                'Std_NoShock': std_0,
                'Std_Shock': std_1,
                'MannWhitney_U': u_stat,
                'p_value_MW': p_value_mw,
                'Cohens_d': cohens_d,
                'Significativa': 'Sí' if p_value_mw < 0.05 else 'No'
            })
        except Exception as e:
            print(f"Error procesando {col}: {e}")
    
    if len(num_results) > 0:
        num_results_df = pd.DataFrame(num_results).sort_values('p_value_MW')
        
        print(f"\nTotal de variables analizadas: {len(num_results_df)}")
        sig_count = (num_results_df['p_value_MW'] < 0.05).sum()
        print(f"Variables con diferencias significativas (p < 0.05): {sig_count}")
        
        print(f"\n✓ Resultados:\n")
        num_results_df
    else:
        print("No se pudieron analizar variables numéricas")
else:
    print("No hay variables numéricas o target no definido")

In [ ]:
# Boxplots de variables numéricas por target
if 'num_results_df' in locals() and not num_results_df.empty and len(numerical_cols) > 0:
    n_num = len(numerical_cols)
    n_cols = min(n_num, 3)
    n_rows = (n_num + n_cols - 1) // n_cols
    
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(6*n_cols, 4*n_rows))
    axes = axes.flatten() if n_rows * n_cols > 1 else [axes]
    
    for idx, col in enumerate(numerical_cols):
        ax = axes[idx]
        
        df_plot = df[[col, target_col]].dropna()
        sns.boxplot(data=df_plot, x=target_col, y=col, ax=ax, palette='Set2')
        ax.set_title(f'{col} por {target_col}', fontsize=12, fontweight='bold')
        ax.set_xlabel(target_col, fontsize=11)
        ax.set_ylabel(col, fontsize=11)
        ax.grid(True, alpha=0.3, axis='y')
    
    # Ocultar ejes sobrantes
    for idx in range(len(numerical_cols), len(axes)):
        axes[idx].axis('off')
    
    plt.tight_layout()
    plt.show()

## 7. Análisis Multivariado

### A. Correlaciones entre Variables Numéricas (Spearman)

In [ ]:
# Matriz de correlación Spearman (numéricas)
if len(numerical_cols) > 1:
    print("="*70)
    print("CORRELACIÓN SPEARMAN - VARIABLES NUMÉRICAS")
    print("="*70)
    
    corr_matrix = df[numerical_cols].corr(method='spearman')
    
    print("\nMatriz de Correlación:\n")
    corr_matrix
else:
    print("No hay suficientes variables numéricas para calcular correlaciones")

In [ ]:
# Heatmap de correlaciones
if len(numerical_cols) > 1:
    fig, ax = plt.subplots(figsize=(10, 8))
    
    mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
    
    sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.2f', 
               cmap='coolwarm', center=0, square=True, ax=ax,
               cbar_kws={"shrink": 0.8}, linewidths=0.5)
    
    ax.set_title('Matriz de Correlación Spearman (Variables Numéricas)', 
                fontsize=14, fontweight='bold')
    
    plt.tight_layout()
    plt.show()

### B. Matriz de Cramer's V (Variables Categóricas)

In [ ]:
# Matriz de Cramer's V (solo top variables por rendimiento)
if len(all_categorical) > 1:
    print("="*70)
    print("MATRIZ DE CRAMER'S V - VARIABLES CATEGÓRICAS")
    print("="*70)
    
    # Seleccionar top 15 variables para no saturar
    top_categorical = all_categorical[:15]
    
    cramers_matrix = pd.DataFrame(
        np.zeros((len(top_categorical), len(top_categorical))),
        index=top_categorical,
        columns=top_categorical
    )
    
    print(f"\nCalculando Cramer's V para {len(top_categorical)} variables...")
    
    for i, col1 in enumerate(top_categorical):
        for j, col2 in enumerate(top_categorical):
            if i <= j:
                v = cramers_v(df[col1], df[col2])
                cramers_matrix.iloc[i, j] = v
                cramers_matrix.iloc[j, i] = v
    
    print("✓ Cálculo completado\n")
    cramers_matrix
else:
    print("No hay suficientes variables categóricas")

In [ ]:
# Heatmap de Cramer's V
if 'cramers_matrix' in locals() and len(cramers_matrix) > 1:
    fig, ax = plt.subplots(figsize=(12, 10))
    
    mask = np.triu(np.ones_like(cramers_matrix, dtype=bool))
    
    sns.heatmap(cramers_matrix, mask=mask, annot=True, fmt='.2f', 
               cmap='YlOrRd', vmin=0, vmax=1, square=True, ax=ax,
               cbar_kws={"shrink": 0.8}, linewidths=0.5)
    
    ax.set_title("Matriz de Cramer's V (Variables Categóricas)", 
                fontsize=14, fontweight='bold')
    
    plt.tight_layout()
    plt.show()

### C. PCA (Análisis de Componentes Principales)

Reducción de dimensionalidad para visualizar la separabilidad de las clases.

In [ ]:
# PCA para reducción de dimensionalidad
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

print("="*70)
print("ANÁLISIS DE COMPONENTES PRINCIPALES (PCA)")
print("="*70)

# Preparar datos: one-hot encoding
df_encoded = pd.get_dummies(
    df.drop(columns=[target_col] + (['CODIGO'] if 'CODIGO' in df.columns else [])),
    drop_first=True
)

# Manejar valores nulos
df_encoded = df_encoded.fillna(df_encoded.median())

print(f"\nDatos codificados: {df_encoded.shape[0]} filas x {df_encoded.shape[1]} columnas")

# Escalar
scaler = StandardScaler()
X_scaled = scaler.fit_transform(df_encoded)

# PCA
n_components = 2
pca = PCA(n_components=n_components)
X_pca = pca.fit_transform(X_scaled)

# Varianza explicada
var_explained = pca.explained_variance_ratio_

print(f"\n✓ PCA completado")
print(f"\nVarianza explicada por componente:")
for i, var in enumerate(var_explained):
    print(f"  • PC{i+1}: {var*100:.2f}%")
print(f"  • Total acumulado: {sum(var_explained)*100:.2f}%")

In [ ]:
# Visualizar PCA en 2D
fig, ax = plt.subplots(figsize=(10, 8))

scatter = ax.scatter(X_pca[:, 0], X_pca[:, 1], 
                    c=df[target_col], cmap='coolwarm', 
                    alpha=0.6, edgecolors='k', linewidth=0.5, s=50)

ax.set_xlabel(f'PC1 ({var_explained[0]*100:.2f}% varianza)', fontsize=12, fontweight='bold')
ax.set_ylabel(f'PC2 ({var_explained[1]*100:.2f}% varianza)', fontsize=12, fontweight='bold')
ax.set_title('Análisis de Componentes Principales (PCA) - Separabilidad de Clases', 
            fontsize=14, fontweight='bold')
ax.grid(True, alpha=0.3)

# Colorbar
cbar = plt.colorbar(scatter, ax=ax)
cbar.set_label(f'Clase ({target_col})', fontsize=12, fontweight='bold')

plt.tight_layout()
plt.show()

**Interpretación del PCA**:
- Si las clases están **bien separadas** → Buen potencial predictivo
- Si las clases están **mezcladas** → Problema complejo, requiere features más discriminativas

## 8. Feature Engineering Exploratorio

Creación de variables compuestas para mejorar el poder predictivo.

In [ ]:
# Identificar variables de comorbilidades
comorbidity_keywords = ['enfermedad', 'diabetes', 'hipertension', 'falla', 'insuficiencia', 
                        'coronaria', 'renal', 'hepatica', 'cardiaca']

comorbidity_cols = [col for col in binary_cols if any(
    keyword in col.lower() for keyword in comorbidity_keywords
)]

print("="*70)
print("FEATURE ENGINEERING EXPLORATORIO")
print("="*70)

print(f"\nVariables de comorbilidades detectadas: {len(comorbidity_cols)}")
if len(comorbidity_cols) > 0:
    print(f"Variables: {comorbidity_cols}")

In [ ]:
# Crear nuevas features
new_features = pd.DataFrame()

if len(comorbidity_cols) > 0:
    # Conteo de comorbilidades
    new_features['n_comorbilidades'] = df[comorbidity_cols].sum(axis=1)
    
    print("\n✓ Variable creada: 'n_comorbilidades'")
    print(f"\nEstadísticas de n_comorbilidades:")
    print(new_features['n_comorbilidades'].describe())
    
    # Análisis vs target
    if target_col:
        print(f"\nDistribución de comorbilidades por {target_col}:")
        print(new_features.groupby(df[target_col])['n_comorbilidades'].describe())

In [ ]:
# Visualizar nueva feature
if 'n_comorbilidades' in new_features.columns and target_col:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Boxplot
    df_temp = pd.DataFrame({
        'n_comorbilidades': new_features['n_comorbilidades'],
        target_col: df[target_col]
    })
    
    sns.boxplot(data=df_temp, x=target_col, y='n_comorbilidades', ax=axes[0], palette='Set2')
    axes[0].set_title('Número de Comorbilidades por Clase', fontsize=12, fontweight='bold')
    axes[0].set_xlabel(target_col, fontsize=11)
    axes[0].set_ylabel('Número de Comorbilidades', fontsize=11)
    axes[0].grid(True, alpha=0.3, axis='y')
    
    # Histograma comparativo
    for val in df[target_col].unique():
        data = new_features[df[target_col] == val]['n_comorbilidades']
        axes[1].hist(data, alpha=0.6, label=f'{target_col}={val}', bins=range(0, int(data.max())+2))
    
    axes[1].set_xlabel('Número de Comorbilidades', fontsize=11)
    axes[1].set_ylabel('Frecuencia', fontsize=11)
    axes[1].set_title('Distribución de Comorbilidades por Clase', fontsize=12, fontweight='bold')
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    # Test estadístico
    if len(df[target_col].unique()) == 2:
        group_0 = new_features[df[target_col] == 0]['n_comorbilidades']
        group_1 = new_features[df[target_col] == 1]['n_comorbilidades']
        
        u_stat, p_value = mannwhitneyu(group_0, group_1, alternative='two-sided')
        
        print(f"\nTest Mann-Whitney U para 'n_comorbilidades':")
        print(f"  • U-statistic: {u_stat:.2f}")
        print(f"  • p-value: {p_value:.4f}")
        print(f"  • Significativo: {'Sí' if p_value < 0.05 else 'No'}")

## 9. Conclusiones y Recomendaciones

### Resumen Ejecutivo

In [ ]:
# Generar resumen ejecutivo
print("="*70)
print("RESUMEN EJECUTIVO DEL EDA")
print("="*70)

print(f"\n📊 **Dataset**: {df.shape[0]:,} registros x {df.shape[1]} variables")

if target_col:
    target_counts = df[target_col].value_counts().sort_index()
    minority = target_counts.min()
    majority = target_counts.max()
    ratio = minority / majority
    
    print(f"\n🎯 **Variable Objetivo**: {target_col}")
    print(f"  • Distribución: {dict(target_counts)}")
    print(f"  • Ratio de desbalance: {ratio:.4f}")
    print(f"  • Estado: {'DESBALANCEADO ⚠️' if ratio < 0.5 else 'BALANCEADO ✓'}")

print(f"\n📝 **Tipificación**:")
print(f"  • Variables numéricas: {len(numerical_cols)}")
print(f"  • Variables binarias: {len(binary_cols)}")
print(f"  • Variables categóricas: {len(categorical_cols)}")

if 'cat_results_df' in locals():
    sig_cat = (cat_results_df['p_value'] < 0.05).sum()
    top_cat_var = cat_results_df.iloc[0]
    print(f"\n🔍 **Asociaciones Categóricas**:")
    print(f"  • Variables significativas: {sig_cat}/{len(cat_results_df)}")
    print(f"  • Variable más asociada: {top_cat_var['Variable']}")
    print(f"    - Cramer's V: {top_cat_var['Cramers_V']:.4f}")
    print(f"    - p-value: {top_cat_var['p_value']:.4e}")

if 'num_results_df' in locals():
    sig_num = (num_results_df['p_value_MW'] < 0.05).sum()
    if len(num_results_df) > 0:
        top_num_var = num_results_df.iloc[0]
        print(f"\n📈 **Diferencias Numéricas**:")
        print(f"  • Variables significativas: {sig_num}/{len(num_results_df)}")
        print(f"  • Variable con mayor diferencia: {top_num_var['Variable']}")
        print(f"    - Cohen's d: {top_num_var['Cohens_d']:.4f}")
        print(f"    - p-value: {top_num_var['p_value_MW']:.4e}")

if 'var_explained' in locals():
    print(f"\n🎨 **PCA**:")
    print(f"  • Varianza explicada (PC1+PC2): {sum(var_explained)*100:.2f}%")
    print(f"  • Separabilidad: {'Moderada' if sum(var_explained) > 0.5 else 'Baja'}")

print("\n" + "="*70)

### Hallazgos Clave

**Variables Significativas Identificadas:**
- Revisar las tablas anteriores para variables con p < 0.05
- Enfocarse en variables con Cramer's V > 0.1 o Cohen's d > 0.2

**Limitaciones del Dataset:**
- Dataset clínico pequeño (~1300 registros)
- Predominancia de variables categóricas/binarias
- Posible desbalance de clases (si aplica)
- Baja varianza explicada en PCA sugiere complejidad

**Recomendaciones para Modelado:**

1. **Manejo de Desbalance**:
   - Aplicar SMOTE o técnicas de oversampling
   - Considerar class_weight='balanced' en modelos

2. **Selección de Features**:
   - Usar variables con asociación significativa (p < 0.05)
   - Considerar feature engineering (ej: n_comorbilidades)
   - Evaluar eliminación de variables redundantes (Cramer's V > 0.8)

3. **Modelos Recomendados**:
   - Logistic Regression (baseline interpretable)
   - Random Forest (maneja categóricas bien)
   - XGBoost (alto rendimiento)
   - Ensemble de modelos

4. **Métricas de Evaluación**:
   - Priorizar Recall (detectar casos positivos de shock)
   - Monitorear Precision y F1-Score
   - Usar AUC-ROC para comparar modelos
   - Considerar especificidad por contexto clínico

5. **Validación**:
   - Cross-validation estratificada (k=5 o k=10)
   - Holdout test set (20-30%)
   - Validación temporal si datos tienen timestamp

### Próximos Pasos

1. ✅ **EDA Completado** - Análisis exploratorio finalizado
2. ⬜ **Preprocesamiento** - Aplicar transformaciones identificadas
3. ⬜ **Feature Engineering** - Implementar nuevas variables
4. ⬜ **Modelado** - Entrenar múltiples algoritmos
5. ⬜ **Evaluación** - Comparar modelos y seleccionar el mejor
6. ⬜ **Calibración** - Ajustar probabilidades predichas
7. ⬜ **Deployment** - Preparar para producción

---

**Fin del Análisis Exploratorio de Datos (EDA)**

## 10. Pipeline Automatizado de EDA

**Bonus**: Este notebook también puede ejecutar el pipeline automatizado completo.

In [ ]:
# Ejecutar pipeline automatizado (opcional)
ejecutar_pipeline_automatizado = False  # Cambiar a True para ejecutar

if ejecutar_pipeline_automatizado:
    print("Ejecutando pipeline automatizado de EDA...")
    
    pipeline = EDA_Pipeline(
        data_path=DATA_PATH,
        output_dir=OUTPUT_DIR
    )
    
    pipeline.run_full_eda()
else:
    print("Pipeline automatizado NO ejecutado.")
    print("Para ejecutarlo, cambiar 'ejecutar_pipeline_automatizado = True'")

---

**Notebook desarrollado por**: Proyecto Shock Hemorrágico  
**Fecha**: 2025  
**Curso**: Procesamiento de Grandes Datos - ICESI  

*Este análisis sigue las mejores prácticas de ciencia de datos y está adaptado específicamente para datasets clínicos pequeños con predominancia de variables categóricas.*